# 004: Word2Vec Embeddings — Skip-gram with negative sampling from scratch

## SKIP-GRAM OBJECTIVE
- Given a center word $w_c$, predict context words $w_o$ within a window.
- **Negative Sampling Loss**: Instead of softmax over entire vocab, sample $k$ negative words:
  $$\mathcal{L} = -\log \sigma(\mathbf{u}_{w_o}^T \mathbf{v}_{w_c}) - \sum_{i=1}^{k} \log \sigma(-\mathbf{u}_{w_i}^T \mathbf{v}_{w_c})$$
---


In [ ]:
import numpy as np

class Word2VecSkipGram:
    """Skip-gram Word2Vec with negative sampling from scratch."""
    def __init__(self, vocab_size, embed_dim=10):
        self.W_center = np.random.randn(vocab_size, embed_dim) * 0.01
        self.W_context = np.random.randn(vocab_size, embed_dim) * 0.01
        self.vocab_size = vocab_size

    def sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def train_pair(self, center_idx, context_idx, neg_indices, lr=0.025):
        v_c = self.W_center[center_idx]
        u_o = self.W_context[context_idx]
        score_pos = self.sigmoid(np.dot(u_o, v_c))
        grad_pos = (score_pos - 1) * v_c
        grad_center = (score_pos - 1) * u_o
        self.W_context[context_idx] -= lr * grad_pos
        for neg_idx in neg_indices:
            u_neg = self.W_context[neg_idx]
            score_neg = self.sigmoid(np.dot(u_neg, v_c))
            self.W_context[neg_idx] -= lr * score_neg * v_c
            grad_center += score_neg * u_neg
        self.W_center[center_idx] -= lr * grad_center



In [ ]:
corpus = "the king loves the queen and the queen loves the king".split()
vocab = sorted(set(corpus))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

model = Word2VecSkipGram(len(vocab), embed_dim=8)
window = 2

for epoch in range(200):
    for i, word in enumerate(corpus):
        center = word2idx[word]
        start = max(0, i - window)
        end = min(len(corpus), i + window + 1)
        for j in range(start, end):
            if j == i:
                continue
            context = word2idx[corpus[j]]
            negatives = np.random.choice([k for k in range(len(vocab)) if k != context], size=3)
            model.train_pair(center, context, negatives, lr=0.05)

print("--- Word Embeddings (first 4 dims) ---")
for w in vocab:
    emb = model.W_center[word2idx[w]][:4]
    print(f"  {w:>6}: {emb.round(3)}")

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

print(f"\ncos(king, queen) = {cosine(model.W_center[word2idx['king']], model.W_center[word2idx['queen']]):.4f}")
print(f"cos(king, and)   = {cosine(model.W_center[word2idx['king']], model.W_center[word2idx['and']]):.4f}")
